# Notebook 00 — Setup SQL Server

Cria o banco de dados `SeguroDB` no SQL Server e popula as 11 tabelas com dados de exemplo a partir dos arquivos CSV locais.

**Pré-requisitos:**
- Container `sqlserver-2022` rodando (`docker compose up -d`)
- ODBC Driver 18 for SQL Server instalado
- Ambiente virtual Python ativado com as dependências do `pyproject.toml`

**Ordem de execução das células: de cima para baixo, uma a uma.**

## 1. Importações e Configuração

In [ ]:
import pyodbc
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

SQL_SERVER   = os.getenv('SQL_SERVER',   'localhost')
SQL_PORT     = os.getenv('SQL_PORT',     '1433')
SQL_USER     = os.getenv('SQL_USER',     'sa')
SQL_PASSWORD = os.getenv('SQL_PASSWORD', 'SqlServer@2022!')

# Localiza a raiz do projeto (onde está o .env) e aponta para data/
PROJECT_ROOT = Path(find_dotenv()).parent
DATA_DIR     = PROJECT_ROOT / 'data'

print(f'Conectando em: {SQL_SERVER}:{SQL_PORT} como {SQL_USER}')
print(f'Dados em    : {DATA_DIR}')

## 2. Criar banco de dados SeguroDB

In [ ]:
conn_str_master = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={SQL_SERVER},{SQL_PORT};'
    f'DATABASE=master;'
    f'UID={SQL_USER};PWD={SQL_PASSWORD};'
    f'TrustServerCertificate=yes;'
)

conn = pyodbc.connect(conn_str_master, autocommit=True)
cursor = conn.cursor()

cursor.execute("""
IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = 'SeguroDB')
    CREATE DATABASE SeguroDB;
""")

print('Banco SeguroDB criado (ou já existia).')
cursor.close()
conn.close()

## 3. Conectar ao SeguroDB e criar tabelas

In [ ]:
conn_str = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={SQL_SERVER},{SQL_PORT};'
    f'DATABASE=SeguroDB;'
    f'UID={SQL_USER};PWD={SQL_PASSWORD};'
    f'TrustServerCertificate=yes;'
)

conn = pyodbc.connect(conn_str, autocommit=True)
cursor = conn.cursor()

ddl_statements = [
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='regiao' AND xtype='U')
    CREATE TABLE dbo.regiao (
        id_regiao   INT PRIMARY KEY,
        nome_regiao VARCHAR(50) NOT NULL
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='estado' AND xtype='U')
    CREATE TABLE dbo.estado (
        id_estado   INT PRIMARY KEY,
        nome_estado VARCHAR(100) NOT NULL,
        uf          CHAR(2) NOT NULL,
        id_regiao   INT REFERENCES dbo.regiao(id_regiao)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='municipio' AND xtype='U')
    CREATE TABLE dbo.municipio (
        id_municipio   INT PRIMARY KEY,
        nome_municipio VARCHAR(150) NOT NULL,
        id_estado      INT REFERENCES dbo.estado(id_estado)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='marca' AND xtype='U')
    CREATE TABLE dbo.marca (
        id_marca    INT PRIMARY KEY,
        nome_marca  VARCHAR(100) NOT NULL,
        pais_origem VARCHAR(50)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='modelo' AND xtype='U')
    CREATE TABLE dbo.modelo (
        id_modelo   INT PRIMARY KEY,
        nome_modelo VARCHAR(100) NOT NULL,
        id_marca    INT REFERENCES dbo.marca(id_marca),
        categoria   VARCHAR(50)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='cliente' AND xtype='U')
    CREATE TABLE dbo.cliente (
        id_cliente      INT PRIMARY KEY,
        nome            VARCHAR(150) NOT NULL,
        cpf             VARCHAR(14) UNIQUE NOT NULL,
        data_nascimento DATE,
        email           VARCHAR(200),
        sexo            CHAR(1)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='endereco' AND xtype='U')
    CREATE TABLE dbo.endereco (
        id_endereco  INT PRIMARY KEY,
        id_cliente   INT REFERENCES dbo.cliente(id_cliente),
        logradouro   VARCHAR(200) NOT NULL,
        numero       VARCHAR(10),
        complemento  VARCHAR(100),
        bairro       VARCHAR(100),
        cep          VARCHAR(10),
        id_municipio INT REFERENCES dbo.municipio(id_municipio)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='telefone' AND xtype='U')
    CREATE TABLE dbo.telefone (
        id_telefone INT PRIMARY KEY,
        id_cliente  INT REFERENCES dbo.cliente(id_cliente),
        ddd         CHAR(2) NOT NULL,
        numero      VARCHAR(15) NOT NULL,
        tipo        VARCHAR(20)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='carro' AND xtype='U')
    CREATE TABLE dbo.carro (
        id_carro        INT PRIMARY KEY,
        id_modelo       INT REFERENCES dbo.modelo(id_modelo),
        placa           VARCHAR(8) UNIQUE NOT NULL,
        chassi          VARCHAR(30) UNIQUE NOT NULL,
        ano_fabricacao  INT,
        cor             VARCHAR(30),
        combustivel     VARCHAR(20)
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='apolice' AND xtype='U')
    CREATE TABLE dbo.apolice (
        id_apolice      INT PRIMARY KEY,
        id_cliente      INT REFERENCES dbo.cliente(id_cliente),
        id_carro        INT REFERENCES dbo.carro(id_carro),
        numero_apolice  VARCHAR(20) UNIQUE NOT NULL,
        data_inicio     DATE NOT NULL,
        data_fim        DATE NOT NULL,
        valor_cobertura DECIMAL(12,2),
        valor_franquia  DECIMAL(10,2),
        status          VARCHAR(20) DEFAULT 'Ativa'
    );
    """,
    """
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='sinistro' AND xtype='U')
    CREATE TABLE dbo.sinistro (
        id_sinistro    INT PRIMARY KEY,
        id_apolice     INT REFERENCES dbo.apolice(id_apolice),
        data_ocorrencia DATE NOT NULL,
        tipo_sinistro  VARCHAR(50),
        descricao      VARCHAR(500),
        valor_prejuizo DECIMAL(12,2),
        status         VARCHAR(30)
    );
    """
]

for stmt in ddl_statements:
    cursor.execute(stmt)

print('Todas as 11 tabelas criadas com sucesso!')

## 4. Carregar dados dos CSVs para o SQL Server

In [ ]:
def insert_dataframe(cursor, table: str, df: pd.DataFrame, chunk_size: int = 500):
    """Insere um DataFrame em uma tabela SQL Server linha a linha em lotes."""
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    sql = f'INSERT INTO dbo.{table} ({cols}) VALUES ({placeholders})'
    
    rows = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]
    
    for i in range(0, len(rows), chunk_size):
        cursor.executemany(sql, rows[i:i+chunk_size])
    
    print(f'  -> {table}: {len(df)} registros inseridos.')


tabelas_ordem = [
    'regiao',
    'estado',
    'municipio',
    'marca',
    'modelo',
    'cliente',
    'endereco',
    'telefone',
    'carro',
    'apolice',
    'sinistro',
]

print('Carregando dados...')
for tabela in tabelas_ordem:
    csv_path = DATA_DIR / f'{tabela}.csv'
    df = pd.read_csv(csv_path)
    # Verifica se tabela já tem dados
    cursor.execute(f'SELECT COUNT(*) FROM dbo.{tabela}')
    count = cursor.fetchone()[0]
    if count > 0:
        print(f'  -> {tabela}: já possui {count} registros, pulando...')
        continue
    insert_dataframe(cursor, tabela, df)

print('\nCarga concluída!')

## 5. Validar carga — contagem de registros por tabela

In [ ]:
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_ordem:
    cursor.execute(f'SELECT COUNT(*) FROM dbo.{tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<15} {count:>10}')

cursor.close()
conn.close()
print('\nConexão encerrada.')